In [ ]:
from analyze_suite2p.config_loader import load_json_config_file

config = load_json_config_file()
print(config.multivid_params)

In [ ]:
from gui_core.io import load_config, save_config
from gui_core import folder_logic
import os

#LOAD Default Config
config = load_config()
config.main_folder = '' #Update main_folder
config.data_extension = '.tif' #Update for your file type

#Add groups / experimental conditions based on your main folder / subfolder organization
config.groups = folder_logic.find_valid_folders(config.main_folder, config.data_extension)
config.exp_condition = folder_logic.build_exp_condition(config.groups)

config.frame_rate = 20 #Adjust frame rate (integer value)
config.ops_path = r'' #Path to suite2p settings file (usually ops.npy)
config.experiment_duration = 180 #(Integer value) for seconds of experiment

#_____ANALYSIS PARAMETERS_____#
config.analysis_params.overwrite_suite2p = 'False'
config.analysis_params.multivid_processing = 'False'
config.analysis_params.baseline_correction = 'airPLS' #options are airPLS or rolling_median
config.analysis_params.lambda_window = 100 #lambda_ value for airPLS or window size for rolling_median dF/F0
config.analysis_params.MAD_baseline_filter_threshold = 2 #Number of MAD-estimated SD to use for filtering peaks from baseline for dF/F0 calculation
config.analysis_params.skew_threshold = 1 #Fluorescence skew threshold for describing trace shape; set to 0 to turn off
config.analysis_params.compactness_threshold = 1.4 #Default value, set to 0 to turn off
config.analysis_params.peak_count_threshold = 1 #Number of peaks per synapse before synapse in included in analysis
config.analysis_params.peak_detection_threshold = 4.5 #Number of SD above MAD to detect peaks; for high SnR, values above 4.5 are possible and could be beneficial
config.analysis_params.Img_Overlay='max_proj'
config.analysis_params.use_suite2p_ROI_classifier=True
config.analysis_params.update_suite2p_iscell=False
config.analysis_params.return_decay_times=False
config.multivid_params.Treatment_No = 2 #Optional parameters for if analysis_params.multivid_processing = True
config.multivid_params.equal_baseline_and_treatments = True #If false, will open the following parameter
config.multivid_params.unequal_treatment_lengths = [] #List of video lengths in frames or seconds
config.multivid_params.treatment_length_units = 'seconds' #options are 'seconds' or 'frames' for converting values in the pipeline

#TO SAVE CONFIGURATIONS
# save_config(path = os.path.join(config.main_folder, 'config', 'config.json'), config=config)

In [ ]:
from analyze_suite2p import plotting_utility, detector_utility, analysis_utility, suite2p_utility, config_loader, run_suite2p
import pandas as pd
import matplotlib.pyplot as plt
import os
import json



config = config_loader.load_json_config_file()
config_dict = config_loader.load_json_dict()


#Run suite2p main
# run_suite2p.main(config_file=config)

#run_suite2p.main() parts
image_folders = run_suite2p.get_all_image_folders_in_path(config.general_settings.main_folder)
suite2p_samples = suite2p_utility.get_all_suite2p_outputs_in_path(folder_path=config.general_settings.main_folder, 
                                                                  config = config, file_ending='samples', 
                                                                  supress_printing=False)

import numpy as np
ops = np.load(config.general_settings.ops_path, allow_pickle=True).item()

run_suite2p.process_files_with_suite2p(image_folders, ops)

main_folder = config.general_settings.main_folder
with open(os.path.join(main_folder, 'analysis_config.json'), 'w') as f:
        json.dump(config_dict, f, indent = 4)
        print(f"Analysis parameters saved in {main_folder} as analysis_config.json")


In [ ]:
#Post processing

analysis_utility.translate_suite2p_outputs_to_csv(main_folder = config.general_settings.main_folder, 
                                                  config = config, 
                                                  check_for_iscell=False, update_iscell=True)
analysis_utility.process_spike_csvs_to_pkl(input_path=config.general_settings.main_folder)


In [ ]:
print(config.general_settings.main_folder)

In [ ]:
print("Exists:", os.path.exists(main_folder))
print("Writable:", os.access(main_folder, os.W_OK))

In [ ]:
aggregate_experiment_summary, experiment_summary = analysis_utility.create_experiment_summary(main_folder=config.general_settings.main_folder)

exp_groups,exp_columns,file_synapses = analysis_utility.generate_synapse_counts_and_summary_stats(main_folder)

experiment = main_folder.split('\\')[-1]
try:
    file_synapses.to_csv(os.path.join(main_folder, f"{experiment}_synapse_count_summary.csv"))
    print("Sucessfully saved .csv file")
except PermissionError as e:
    print(e)
    experiment = main_folder.split('/')[-1]
    file_synapses.to_csv(os.path.join(main_folder, f"{experiment}_synapse_count_summary.csv"))
    print("Sucessfully saved .csv file")

In [ ]:
suite2p_files = suite2p_utility.get_all_suite2p_outputs_in_path(main_folder, file_ending = 'samples',config = config, supress_printing=False)

#Example file loading
suite2p_dict = suite2p_utility.load_suite2p_output(suite2p_files[2], config = config, use_iscell = False)

In [ ]:
plotting_utility.plot_synapse_traces(suite2p_dict=suite2p_dict, frame_rate = 20, trace_offset=2.5, list = None,
                                     treatment_vid = False, treatment_no=0, show_peaks = False, 
                                     save_fig = False)

In [ ]:
from analyze_suite2p import plotting_utility, detector_utility, analysis_utility, suite2p_utility, config_loader
import pandas as pd
import matplotlib.pyplot as plt
import os

config = config_loader.load_json_config_file()
main_folder = config.general_settings.main_folder
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ["Arial"]
color_blind_palette = [
    "#E69F00", "#56B4E9", "#009E73",
    "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#999999"]

pkl_paths,pkl_files = plotting_utility.get_all_pkl_outputs_in_path(main_folder)
print(pkl_paths, pkl_files)

for i in range(1,len(pkl_paths)+1):
    plotting_utility.pynapple_plots(file_path = pkl_paths[i], 
                                    output_directory= os.path.join(main_folder, 'rasterplots'),
                                    treatment_vid=False, treatment_no = 0, synapse_count = None, max_amplitude=None,
                                    plot_amplitudes = True, plot_shape = 'rectangle', save_fig=False)